**Bronze Helpers - Formula 1 Incremental**

Shared functions used by all bronze ingestion notebooks.

- `add_ingestion_metadata(df)` — adds `ingestion_timestamp` and `source_file` columns
- `write_to_bronze(input_df, table_name, batch_id)` — adds `batch_id`, partitions by it, and writes to Delta table with overwrite per batch

In [0]:
# Helpers function to add the file metadata and ingestion timestamp

from pyspark.sql.functions import *

def add_ingestion_metadata(df):
    return(df
        .withColumn('ingestion_timestamp', current_timestamp())
        .withColumn('source_file', col('_metadata.file_path'))
    )

In [0]:
def write_to_bronze (
    input_df,
    table_name,
    batch_id
):
    final_df = input_df.withColumn('batch_id', f.lit(batch_id))
    (
        final_df
         .write
         .format('delta')
         .mode('overwrite')
         .option('overwriteSchema', 'true')
         .partitionBy('batch_id')
         .option('replaceWhen',f'batch_id = "{batch_id}"')
         .saveAsTable(table_name)
      
      )
